In [12]:
from datetime import date
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

url = "https://drive.google.com/uc?id=1u2DE9aWZrLfdl3ZpHtiLa4RPO8RmgnHY"

df = pd.read_csv(url)

# Fix: Rename the Transaction_ID column to remove the BOM character
if 'ï»¿Transaction_ID' in df.columns:
    df.rename(columns={'ï»¿Transaction_ID': 'Transaction_ID'}, inplace=True)

# Convert date - Moved this line after df is loaded
df["Order_Date"] = pd.to_datetime(df["Order_Date"], errors="coerce")

df.convert_dtypes()
print(df.dtypes)


# Remove duplicates
df = df.drop_duplicates()

# Handle missing values
df["Total_Amount"] = pd.to_numeric(df["Total_Amount"], errors="coerce")
df["Age"] = pd.to_numeric(df["Age"], errors="coerce")

df.dropna(subset=["Customer_ID", "Order_Date"], inplace=True)

#RFM ANALYSIS
reference_date = df["Order_Date"].max()

rfm = df.groupby("Customer_ID").agg({
    "Order_Date": lambda x: (reference_date - x.max()).days,
    "Transaction_ID": "count", # This will now work correctly after renaming
    "Total_Amount": "sum"
})

rfm.columns = ["Recency", "Frequency", "Monetary"]

print(rfm.head())





/tmp/ipykernel_1090/2738432580.py:15: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df["Order_Date"] = pd.to_datetime(df["Order_Date"], errors="coerce")


Transaction_ID             float64
Customer_ID                float64
Customer_Name               object
City                        object
State                       object
Country                     object
Age                        float64
Gender                      object
Order_Date          datetime64[ns]
Product_Name                object
Quantity                   float64
Unit_Price                 float64
Total_Amount               float64
Product_Category            object
Product_Brand               object
dtype: object
             Recency  Frequency     Monetary
Customer_ID                                 
10000.0          379          4  5007.566467
10001.0          381          5  8136.462738
10002.0          371          5  4104.014053
10003.0          504          2  2340.496399
10004.0          307          2  2356.516663


In [13]:
#customer segmentation
rfm["R_score"] = pd.qcut(rfm["Recency"], 5, labels=[5,4,3,2,1]).astype(int)
rfm["F_score"] = pd.qcut(rfm["Frequency"].rank(method="first"), 5, labels=[1,2,3,4,5]).astype(int)
rfm["M_score"] = pd.qcut(rfm["Monetary"], 5, labels=[1,2,3,4,5]).astype(int)

rfm["RFM_Score"] = (
    rfm["R_score"].astype(str) +
    rfm["F_score"].astype(str) +
    rfm["M_score"].astype(str)
)

def segment(row):
    if row["RFM_Score"] == "555":
        return "Best Customers"
    elif row["R_score"] >= 4 and row["F_score"] >= 4:
        return "Loyal Customers"
    elif row["R_score"] >= 4:
        return "Recent Customers"
    elif row["F_score"] >= 4:
        return "Frequent Customers"
    else:
        return "At Risk"

rfm["Segment"] = rfm.apply(segment, axis=1)



In [14]:
#product analysis
# Revenue by category
print(df.groupby("Product_Category")["Total_Amount"].sum().sort_values(ascending=False))

# Top products
print(df["Product_Name"].value_counts().head(10))

# Brand performance
print(df.groupby("Product_Brand")["Total_Amount"].sum().sort_values(ascending=False))



Product_Category
Electronics    9.713090e+07
Grocery        9.084793e+07
Clothing       7.451651e+07
Books          7.431309e+07
Home Decor     7.395303e+07
Name: Total_Amount, dtype: float64
Product_Name
Spring water       2500
Bottled water      2491
Mystery            2489
Artesian water     2472
Distilled water    2467
Alkaline water     2446
Adventure          2431
Mineral water      2424
Flavored water     2419
Coconut water      2418
Name: count, dtype: int64
Product_Brand
Pepsi                4.128093e+07
Samsung              2.528756e+07
Sony                 2.498136e+07
Nike                 2.495148e+07
Coca-Cola            2.488385e+07
Zara                 2.484364e+07
HarperCollins        2.482160e+07
Bed Bath & Beyond    2.480248e+07
Penguin Books        2.475848e+07
Home Depot           2.475528e+07
Adidas               2.471812e+07
Random House         2.471248e+07
Nestle               2.469734e+07
Apple                2.453229e+07
IKEA                 2.441362e+07
Whire

In [15]:
# customer behaviour analysis.

# Revenue by segment
print(rfm.groupby("Segment")["Monetary"].sum())

# Customer count by segment
print(rfm["Segment"].value_counts())

Segment
At Risk               1.088587e+08
Best Customers        3.763186e+07
Frequent Customers    1.056171e+08
Loyal Customers       1.029825e+08
Recent Customers      5.645782e+07
Name: Monetary, dtype: float64
Segment
At Risk               36231
Recent Customers      15821
Loyal Customers       15631
Frequent Customers    15451
Best Customers         3619
Name: count, dtype: int64


In [18]:
#demographic analysis
# Age vs spending
print(df.groupby("Age")["Total_Amount"].mean())

# Gender vs spending
print(df.groupby("Gender")["Total_Amount"].mean())

# City vs revenue
print(df.groupby("City")["Total_Amount"].sum().sort_values(ascending=False))

Age
18.0     770.656372
19.0    1278.196873
20.0    1328.973262
21.0    1384.117747
22.0    1374.885849
23.0    1368.020425
24.0    1381.894112
25.0    1375.622424
26.0    1359.013568
27.0    1411.869475
28.0    1419.835192
29.0    1422.367232
30.0    1393.574326
31.0    1412.494564
32.0    1383.584927
33.0    1419.453359
34.0    1373.384923
35.0    1394.159534
36.0    1388.429566
37.0    1376.264228
38.0    1374.992932
39.0    1454.285270
40.0    1412.091385
41.0    1398.352649
42.0    1391.101317
43.0    1422.826375
44.0    1398.335161
45.0    1395.725677
46.0    1382.387081
47.0    1385.659587
48.0    1383.386247
49.0    1403.477295
50.0    1384.189754
51.0    1442.498516
52.0    1417.545036
53.0    1398.272744
54.0    1398.031720
55.0    1382.796696
56.0    1419.547079
57.0    1389.733030
58.0    1406.869019
59.0    1407.584629
60.0    1388.635775
61.0    1397.755496
62.0    1411.043060
63.0    1368.202592
64.0    1413.995208
65.0    1379.283956
66.0    1379.164516
67.0    1367.750

In [16]:
#LOGISTIC REGRESSSION
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

# Create target (loyal)
rfm["Loyal"] = rfm["RFM_Score"].apply(
    lambda x: 1 if x in ["555", "554", "545"] else 0
)

X = rfm[["Recency", "Frequency", "Monetary"]]
y = rfm["Loyal"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,random_state=42)

model = LogisticRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print(y_pred)

[1 0 0 ... 1 0 0]


In [17]:
#EVALUATE THE MODEL
from sklearn.metrics import accuracy_score, classification_report

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.9522217739611549
              precision    recall  f1-score   support

           0       0.96      0.99      0.97     16079
           1       0.75      0.53      0.62      1272

    accuracy                           0.95     17351
   macro avg       0.85      0.76      0.80     17351
weighted avg       0.95      0.95      0.95     17351

